# 02 — ReMedFiLM historical component training

Recovers the actual training evidence for each of the four components, and documents how the FULL system was assembled from them.

**Evidence base**

- `full_doc_report/tables/TRAINING_RUN_REGISTRY.csv`
- `experiments/phase21_global_medfilm_fourdomain_seed42_clean_v3/closure.json`
- `experiments/phase25*/**/run.json`

> Execution is `ANALYSIS_ONLY`. Training, new inference and protected TEST access
> are blocked by an installed audit hook (verified below).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, r"D:\Softfors\jup\full_doc_report\Scripts")
import remedfilm_guard as GUARD
GUARD.install()

REPO = Path(r"D:\Softfors\jup")
REPORT = REPO / "full_doc_report"
TABLES = REPORT / "tables"
FIG = REPORT / "figures"

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

def fig(subdir, name, width=760):
    """Display a generated figure, failing loudly if it is missing."""
    p = FIG / subdir / f"{name}.png"
    if not p.exists():
        raise FileNotFoundError(f"missing figure: {p}")
    display(Image(filename=str(p), width=width))

def table(path, **kw):
    return pd.read_csv(path, **kw)

GUARD.require()

True

In [2]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified


## 2.1 Backbone G — the only component trained on all four domains

In [3]:
cl = json.loads((REPO / "experiments/phase21_global_medfilm_fourdomain_seed42_clean_v3/closure.json").read_text(encoding="utf-8"))
cfg = cl["configuration"]
print("experiment      :", cl["experiment_id"])
print("population      :", cfg["dataset"])
print("model           :", cfg["model"]["name"], "| params", cl["model_params"])
print("epochs          :", cl["total_completed_epochs"], "| best", cl["best_epoch"])
print("stop reason     :", cl["stop_reason"], f"(patience {cl['early_stopping_patience']})")
print("optimizer       :", cfg["optimizer"])
print("scheduler       :", cfg["scheduler"])
print("loss            :", cfg["loss"])
print("batch / amp     :", cfg["training"]["batch_size"], "/", cfg["training"]["amp"])
print("checkpoint rule :", cfg["training"]["checkpoint_metric"])
print("best ckpt sha   :", cl["best_checkpoint_sha256"])
print("wall clock (s)  :", round(cl["total_elapsed_seconds"], 1))

experiment      : phase21_global_medfilm_fourdomain_seed42_clean_v3
population      : MULTI_DOMAIN_ISIC_BUSI_KVASIR_KUB
model           : GlobalMedFiLM | params 3523001
epochs          : 81 | best 66
stop reason     : EARLY_STOPPING_PATIENCE (patience 15)
optimizer       : {'name': 'AdamW', 'learning_rate': 0.0001, 'weight_decay': 0.0001}
scheduler       : {'name': 'CosineAnnealingLR', 'T_max': 100, 'step': 'per_epoch_after_train_and_val'}
loss            : {'name': 'bce_dice', 'bce_weight': 0.5, 'dice_weight': 0.5, 'dice_epsilon': 1e-06}
batch / amp     : 4 / True
checkpoint rule : phase20_macro_domain_mean_dice_v1
best ckpt sha   : 286689fdd5c63ad75a383151d267af8b476e7a98327632de292ce6ed5b5ba1b7
wall clock (s)  : 34588.8


## 2.2 Component runs — registry view

In [4]:
reg = pd.read_csv(TABLES / "TRAINING_RUN_REGISTRY.csv")
comp = reg[reg.model_role.isin(["BACKBONE", "COMPONENT"])]
cols = ["run_id", "model", "seed", "epochs_completed", "best_epoch",
        "best_epoch_source", "evidence_status"]
print("component runs:", len(comp))
comp[cols].head(24)

component runs: 52


,run_id,model,seed,epochs_completed,best_epoch,best_epoch_source,evidence_status
22,phase25a_barr_regional_refiner_v1_1__final_refit,R_psi (Regional Refiner),42,27,NOT_APPLICABLE,NOT_APPLICABLE_FIXED_EPOCH_REFIT,GENUINE_HISTORICAL_TRAINING
23,phase25a_barr_regional_refiner_v1_1__fold0,R_psi (Regional Refiner),42,50,34,RECORDED_CLOSURE,GENUINE_HISTORICAL_TRAINING
24,phase25a_barr_regional_refiner_v1_1__fold1,R_psi (Regional Refiner),42,43,27,RECORDED_CLOSURE,GENUINE_HISTORICAL_TRAINING
25,phase25a_barr_regional_refiner_v1_1__fold2,R_psi (Regional Refiner),42,34,18,RECORDED_CLOSURE,GENUINE_HISTORICAL_TRAINING
26,phase25a_barr_regional_refiner_v1_1_roi030__fi...,R_psi (Regional Refiner),42,25,NOT_APPLICABLE,NOT_APPLICABLE_FIXED_EPOCH_REFIT,GENUINE_HISTORICAL_TRAINING
27,phase25a_barr_regional_refiner_v1_1_roi030__fold0,R_psi (Regional Refiner),42,41,25,RECORDED_CLOSURE,GENUINE_HISTORICAL_TRAINING
28,phase25a_barr_regional_refiner_v1_1_roi030__fold1,R_psi (Regional Refiner),42,36,20,RECORDED_CLOSURE,GENUINE_HISTORICAL_TRAINING
29,phase25a_barr_regional_refiner_v1_1_roi030__fold2,R_psi (Regional Refiner),42,41,25,RECORDED_CLOSURE,GENUINE_HISTORICAL_TRAINING
30,phase25d_barr_benefit_router_v1_2_1_seed42__final,B_phi (Benefit Router),42,80,NOT_RECORDED,NOT_RECORDED,GENUINE_HISTORY_BEST_EPOCH_UNRESOLVED
31,phase25d_barr_benefit_router_v1_2_1_seed42__fold0,R_psi (Regional Refiner),42,57,37,RECORDED_FLAG,GENUINE_HISTORICAL_TRAINING


In [5]:
# Per-component summary: how many runs, how many epochs, which metrics.
hist = pd.read_csv(TABLES / "REMEDFILM_EPOCH_HISTORY.csv")
summary = (hist.groupby("model_identity")
               .agg(runs=("run_id", "nunique"),
                    epoch_records=("metric", "size"),
                    metrics=("metric", "nunique"))
               .sort_values("epoch_records", ascending=False))
summary

,runs,epoch_records,metrics
model_identity,,,
R_psi (Regional Refiner),21,6476,9
B_phi (Benefit Router),13,4649,4
KUB baseline,6,4349,11
S_omega (Post-Safety Gate),18,3407,4
U-Net,3,3050,31
KUB localizer (ROI proposal),8,2592,9
Transformer/MiT-style U-Net,1,1958,22
GlobalMedFiLM (G),1,1863,23
Attention U-Net,1,1738,22


## 2.3 Checkpoint identity per component

In [6]:
for sym in ("G", "R_psi", "B_phi", "S_omega"):
    sub = reg[reg.model.str.startswith(sym.split("_")[0])]
    ck = sub[sub.best_checkpoint != "NOT_FOUND"]
    print(f"{sym:8s} runs={len(sub):3d}  with_checkpoint={len(ck):3d}")
    if len(ck):
        r = ck.iloc[-1]
        print(f"         e.g. {r.best_checkpoint}")
        print(f"              sha256 {r.checkpoint_hash[:32]}...")

G        runs=  1  with_checkpoint=  0
R_psi    runs= 22  with_checkpoint=  0
B_phi    runs= 13  with_checkpoint=  0
S_omega  runs= 18  with_checkpoint=  0


## 2.4 Dependency chain

In [7]:
print("""
G   (Phase 21)  TRAIN5832 -> VAL808 selection        best epoch 66 / 81
 |
 +-> R_psi (25A)  ROI crops, 3 cross-fitted folds + final refit
      |
      +-> B_phi (25D)  OOF gain labels, 5 CV folds + final
           |
           +-> S_omega (25F)  routed candidates, 5 CV folds + final
                |
                +-> assembly (Phase 26) -> evaluation (27/29/30)
""")
print("Each arrow means the downstream module consumed a FROZEN upstream output.")
print("Nothing in this chain is joint end-to-end training.")


G   (Phase 21)  TRAIN5832 -> VAL808 selection        best epoch 66 / 81
 |
 +-> R_psi (25A)  ROI crops, 3 cross-fitted folds + final refit
      |
      +-> B_phi (25D)  OOF gain labels, 5 CV folds + final
           |
           +-> S_omega (25F)  routed candidates, 5 CV folds + final
                |
                +-> assembly (Phase 26) -> evaluation (27/29/30)

Each arrow means the downstream module consumed a FROZEN upstream output.
Nothing in this chain is joint end-to-end training.


## 2.5 What was *not* recorded

| Field | Status |
|---|---|
| Per-epoch wall clock for R/B/S | partially recorded (`elapsed_seconds`) |
| Peak GPU memory for G | **NOT_RECORDED** in the closure |
| Independent duplicate runs of G | 0 (`independent_duplicate_runs`) |
| Three independent end-to-end seeds | **NOT_EXECUTED** — only downstream components were re-seeded |

In [8]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified
